# Anomaly detection from normal batches only (second solution, not scored)

**Objective.** Identify batches that deviate from normal behaviour, using only accepted batches for learning. No rejected batch and no label is used.

**Method.**
1. The hybrid model v7 predicts every labelled batch out-of-fold; the residual `log(measured / predicted)` describes the normal deviation of a batch from its forecast.
2. Each batch is compared with the residuals of the other 48 batches (leave-one-out).
3. Two kinds of detector are used: a batch-level score (D1, D2) and cell-level rules (spikes and persistent deviations).
4. No rejected batches exist, so detection performance is estimated with injected faults, and the false-alarm rate with the normal batches.
5. The 12 test batches have no measurements; only their inputs can be checked.

**Inputs.** The v7 files of branch `hybrid_v3_tr` (`members_oof_v7c.csv`, `blend_weights.csv`), read as committed at `hybrid_v7.ref` in `params.yaml`, and the event log in `data/raw/`.

**Code.** The computations are in `bioreactor_decoding/anomaly.py`, unchanged from the original notebook (branch `anomalies_detection`); this notebook calls them. The DVC stage `anomaly` (`python -m bioreactor_decoding.anomaly`) writes the same tables to `reports/anomaly/`, the summary to `reports/metrics/anomaly.json` and the figures to `reports/figures/`.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from bioreactor_decoding import anomaly as an

T = an.T
MEM = an.MEM          # the four members of hybrid v7

## 1. Residuals of the normal-behaviour model

The out-of-fold predictions of the four v7 members (7-fold cross-validation over whole batches, so no prediction has seen its own batch) are combined with the v7 blend weights (weighted geometric mean, one set per variable and day range). The finishing steps of v7 are not repeated. The residual of a batch is the vector of 52 values `log(measured / predicted)` (4 variables, 13 days).

In [ ]:
m, W = an.load_v7()                                      # as committed at hybrid_v7.ref
P = an.blend(m, W)

cv = an.blend_mape(m, P)
print("MAPE of the blend (%):", {k: round(x, 2) for k, x in cv.items()}, "mean", round(np.mean(list(cv.values())), 2))

ids, Obs, Pred, R, RV = an.residuals(m, P)               # 49 batches x 52 residuals
print(f"typical spread of a residual (log units): {R.std(0).mean():.3f}")

**Interpretation.** The blend reproduces the cross-validated error of v7 (7.16 %). The typical spread of a residual is 0.096 in log units, i.e. about 10 %. This spread is the normal deviation of a batch from its forecast and therefore sets the smallest deviation that can be detected.

## 2. Batch-level detectors

For each batch, the normal model is fitted on the other 48 batches. Two detectors are computed:

* **D1:** root mean square of the robust z-values (median and MAD of the other batches, each z clipped at 6) of the 52 residuals.
* **D2:** PCA of the normal residual patterns (3 components): **T²** measures the distance inside the components, **SPE** the distance outside them.

A batch is flagged if any of the three scores exceeds its 95 % quantile over the normal batches (5 % false alarms per score).

In [ ]:
batch, THR = an.batch_scores(R, ids)
flag = batch["flagged"].to_numpy()
print(f"thresholds: D1 {THR[0]:.2f}, T2 {THR[1]:.2f}, SPE {THR[2]:.1f}; flagged batches: {int(flag.sum())} of {len(ids)}")
display(batch.sort_values("D1", ascending=False).head(6).round(2))

**Interpretation.** Six of the 49 batches are flagged by at least one of the three scores. The scores of the highest-ranked batches lie close to the thresholds (D1 1.87 against 1.79), and the three scores flag different batches. No batch is a clear outlier at the level of the whole curve.

## 3. Cell-level rules

A batch score averages over 52 values, so a single extreme value is diluted. Two rules therefore test individual values. The z-value of a value is its distance, in robust standard deviations, from the residuals of the same variable and day in the other 48 batches.

| rule | definition | interpretation |
|---|---|---|
| spike | one value with absolute z of at least *s*, neighbouring days below *p* | probable measurement or sampling artefact |
| persistent deviation | absolute z of at least *p*, same sign, for at least 3 consecutive days (VCD, substrate, byproduct) | probable process difference |

Two levels are defined: **standard** (*s* = 5.5, *p* = 3.0) and **strict** (*s* = 6.5, *p* = 3.5). The thresholds were selected so that the standard level flags about 10 % of the normal batches, the level of the batch score. Looser thresholds flagged one quarter to one third of the batches.

In [ ]:
dev = an.cell_deviations(RV, ids)
display(dev)
print(f"standard level: {dev.batch.nunique()} of {len(ids)} batches flagged")

In [ ]:
an.cell_level_figure(Obs, Pred, ids); plt.show()

**Interpretation.**

* The standard level flags 5 of 49 batches: two spikes (Batch34 and Batch29, byproduct on day 13, +270 % and +242 %) and three persistent deviations (Batch46 glucose -29 % on days 4-7, Batch4 lactate -22 % on days 2-6, Batch50 glucose +22 % on days 4-6). The strict level retains Batch34, Batch29 and Batch46.
* The two day-13 values rise from about 0.5 to 1.7-1.9 within one day, which is implausible for cell metabolism; a measurement or sampling problem is more likely.
* The flagged share (10 %) equals the false-alarm rate that the thresholds were set to give. The flagged batches are therefore candidates, not confirmed anomalies. The classification into artefacts and process differences is an interpretation to be confirmed by a process expert.

## 4. Evaluation with injected faults

No rejected batch is available. Faults are therefore injected into the residuals of normal batches, and the share of faulty batches that raise an alarm is measured. Every batch is judged against the normal model of the other 48 batches. The row *no fault* gives the false-alarm rate.

| fault | change applied | size (log change at the end) |
|---|---|---|
| growth stall | cells decrease, glucose increases, from day 4 | 0.1, 0.2, 0.3, 0.5 (about 10, 22, 35, 65 %) |
| lactate overproduction | byproduct increases, from day 3 | same |
| glucose feed failure | glucose decreases from day 8 | same |
| single assay outlier | one random value (variable, day) is shifted | same |

In [ ]:
tab = an.fault_table(R, RV, THR, ids)
display(tab.round(2))

In [ ]:
an.detection_figure(tab); plt.show()

**Interpretation.**

* False-alarm rate on normal batches: 12 % (batch score), 10 % (cell rules, standard), 6 % (cell rules, strict).
* Faults of about 65 % (size 0.5) are detected in most cases: growth stall 98 / 94 / 84 %, glucose feed failure 94 / 98 / 98 %, single wrong value 39 / 76 / 63 %, lactate overproduction 53 / 65 / 39 % (batch score / standard / strict). Faults of 10-20 % are detected at about the false-alarm level and are therefore not distinguishable from normal variation.
* The detectors are complementary. The cell rules are better for local faults (glucose failure of size 0.3: 78 % against 39 %), the batch score for the diffuse growth stall (size 0.3: 65 % against 47 %).
* The faults are synthetic and of simple shape, and the cell-rule thresholds were selected on the same 49 batches, so the false-alarm rates are slightly optimistic.

## 5. Share of real alarms when bad batches are rare

The share of alarms that correspond to a real problem depends on the proportion of bad batches. With the false-alarm rate (FPR) and the average detection rate (TPR) of faults of size 0.5 from section 4:

`real alarms = 1000 · p · TPR`, `false alarms = 1000 · (1 − p) · FPR`, for *p* the proportion of bad batches.

In [ ]:
display(an.alarm_share(tab))

**Interpretation.** If 0.1 % of the batches are bad, about 1 alarm in 100 is real for every detector. At 1 % the share is 6-10 %, at 5 % it is 23-38 %. The strict level gives the highest share of real alarms (10 % and 38 %) at the same detection as the batch score (71 %). The detector is therefore suitable as a screening tool when bad batches are not extremely rare; for a proportion of about 0.1 % the false-alarm rate would have to be reduced to 0.1-1 %, which would restrict detection to very large faults.

## 6. Input check of the 12 test batches

The residual-based checks need measurements, which are hidden for the test batches. Their inputs are available: day-0 values, feeds, volumes and the soft sensors derived from them (unlogged base volume, bolus size per culture volume, as in v3). Each test batch is compared with the 49 labelled batches by (i) the largest z-value of any feature, (ii) the training range of each feature, (iii) a PCA score (T², SPE; 4 components; thresholds at the 95 % quantile of the leave-one-out scores of the labelled batches).

In [ ]:
feat, train, test = an.input_features()
inputs, thr2, top3 = an.input_check(feat, train, test)
display(inputs)
print(f"thresholds: T2 {thr2[0]:.1f}, SPE {thr2[1]:.1f}")
print("largest abs z among the labelled batches (top 3):", top3)

**Interpretation.** Batch54 is the most unusual test batch: its day-0 glucose is 3.8 standard deviations from the mean (the same z-value as the most extreme labelled batch, Batch23) and outside the training range, and its SPE (20.5) exceeds the threshold (18.5). Batch5 is flagged marginally (T2 7.8 against 7.6; day-0 lactate above the training range); day-0 lactate is also above the training range in Batch40 and Batch57. The remaining test batches lie within the training range for nearly all features. With 12 batches and 17 features a few values outside the range are expected by chance. The check concerns inputs only and gives no information about the outcome of a batch.

## 7. Conclusions and limitations

**Conclusions**

* A model of normal behaviour fitted on accepted batches only (v7 forecast and residual spread) supports two complementary checks: cell-level rules for local deviations and a batch score for diffuse ones. Large deviations (about 50 % and more) are detected reliably; deviations of 10-20 % are not.
* In the 49 training batches, two one-day spikes and three persistent deviations of glucose or lactate are identified. They are candidates for review, not confirmed anomalies.
* The detector is a screening tool: an alarm indicates that a person should examine the batch. Its practical value depends on the proportion of bad batches and the cost of a missed one (section 5).

**Limitations**

* No rejected batch is available; detection rates are based on injected faults.
* Thresholds are estimated on 49 batches and are noisy.
* The test batches can be checked on their inputs only. Each test batch contributes 1/12 of the leaderboard score, so one anomalous batch with an average error of 30 % would add about 2.5 points to the error of each variable concerned; such a batch cannot be predicted from day 0 and the log if its cause is not recorded there.
* The residual-based checks depend on out-of-fold v7 predictions; a deployed version would require regular re-fitting and day-by-day evaluation using only data available up to each day.